# COMPAS API 데이터 번호 탐색

`SBJ_2601_001` 과제의 데이터 번호를 1~50번까지 순차적으로 시도하여
어떤 파일이 제공되는지 확인합니다.

**이미 알고 있는 번호:**
| 번호 | 파일 |
|------|------|
| 4 | 04._성연령별_유동인구.csv |
| 5 | 05._시간대별_직장인구.csv |
| 6 | 06._시간대별_방문인구.csv |
| 7 | 07._주중주말_서비스인구.csv |
| 8 | 13._교통사고이력.geojson |
| 24 | 08.상세도로망_네트워크.geojson |
| 25 | 09._평균속도.csv |
| 26 | 10._추정교통량.csv |
| 27 | 11._혼잡빈도강도.csv |
| 28 | 12._혼잡시간강도.csv |

In [ ]:
import pathlib
import shutil
import os
from geoband import API

SUBJECT_CODE = 'SBJ_2601_001'
SCAN_RANGE   = range(1, 51)       # 1~50번 탐색
SCAN_DIR     = pathlib.Path('./api_scan_tmp')
SCAN_DIR.mkdir(exist_ok=True)

# 이미 알고 있는 번호 (건너뜀)
KNOWN = {4, 5, 6, 7, 8, 24, 25, 26, 27, 28}

print(f"탐색 범위: {SCAN_RANGE.start} ~ {SCAN_RANGE.stop - 1}")
print(f"건너뛸 번호 (기존 확인): {sorted(KNOWN)}")
print("-" * 50)

In [ ]:
results = {}   # {번호: 파일명 or None}

for num in SCAN_RANGE:
    if num in KNOWN:
        results[num] = '(기존 확인 완료)'
        continue

    tmp_name = f'_scan_{num}_tmp'
    try:
        API.GetCompasData(SUBJECT_CODE, str(num), tmp_name)
        # 루트에 저장된 파일 탐색
        found = list(pathlib.Path('.').glob(f'{tmp_name}*'))
        if found:
            actual = found[0]
            # scan 폴더로 이동 (파일명 보존)
            dest = SCAN_DIR / actual.name
            shutil.move(str(actual), str(dest))
            results[num] = actual.name
            print(f"  [{num:>2}] ✅ {actual.name}")
        else:
            results[num] = None
            print(f"  [{num:>2}] ❌ 파일 없음")
    except Exception as e:
        results[num] = None
        print(f"  [{num:>2}] ❌ 오류: {e}")

In [ ]:
print("\n" + "=" * 60)
print("탐색 결과 요약")
print("=" * 60)

available = {k: v for k, v in results.items() if v and v != '(기존 확인 완료)'}
print(f"\n새로 발견된 데이터: {len(available)}개")
for num, fname in sorted(available.items()):
    print(f"  {num:>3}번 → {fname}")

print(f"\n다운로드된 파일 목록 ({SCAN_DIR}):")
for p in sorted(SCAN_DIR.glob('*')):
    size = p.stat().st_size / 1024
    print(f"  {p.name}  ({size:.0f} KB)")

## 필요한 파일과 대조

아래 셀에서 `data/` 폴더의 파일과 API로 받은 파일을 비교합니다.

In [ ]:
import json

def get_header(path):
    """파일 첫 줄(컬럼) 추출 - CSV/GeoJSON 모두 지원"""
    try:
        # GeoJSON
        if path.suffix == '' or str(path).endswith('.geojson') or str(path).endswith('.tmp'):
            with open(path, 'r', encoding='utf-8-sig', errors='ignore') as f:
                head = f.read(500)
            if head.strip().startswith('{'):
                # GeoJSON - properties 키 추출
                data = json.loads(open(path, encoding='utf-8-sig').read())
                if 'features' in data and data['features']:
                    return list(data['features'][0].get('properties', {}).keys())
                return ['(GeoJSON - properties 없음)']
        # CSV
        with open(path, 'r', encoding='utf-8-sig', errors='ignore') as f:
            first_line = f.readline().strip()
        return first_line.split(',')[:8]   # 첫 8개 컬럼만
    except Exception as e:
        return [f'(읽기 실패: {e})']

def get_size_kb(path):
    return path.stat().st_size / 1024

# data/ 폴더 파일 정보
data_dir = pathlib.Path('./data')
data_files = {}
for p in sorted(data_dir.glob('*')):
    data_files[p.name] = {
        'path':   p,
        'size':   get_size_kb(p),
        'header': get_header(p),
    }

# scan 파일 정보
scan_files = {}
for p in sorted(SCAN_DIR.glob('*')):
    num = p.name.replace('_scan_', '').replace('_tmp', '')
    scan_files[num] = {
        'path':   p,
        'size':   get_size_kb(p),
        'header': get_header(p),
    }

print("=" * 70)
print("API 번호 ↔ data/ 파일 매칭 결과 (크기 + 컬럼 비교)")
print("=" * 70)

matched_data = set()

for num, sinfo in sorted(scan_files.items(), key=lambda x: int(x[0])):
    s_size   = sinfo['size']
    s_header = sinfo['header']

    best_match = None
    best_score = 0

    for fname, dinfo in data_files.items():
        d_size   = dinfo['size']
        d_header = dinfo['header']

        # 크기 유사도: 10% 이내면 유사
        size_ok = abs(s_size - d_size) / max(d_size, 1) < 0.1

        # 컬럼 일치 수
        s_set = set(str(c).strip() for c in s_header)
        d_set = set(str(c).strip() for c in d_header)
        col_match = len(s_set & d_set)

        score = (2 if size_ok else 0) + col_match
        if score > best_score:
            best_score = score
            best_match = fname

    if best_score >= 2:
        matched_data.add(best_match)
        print(f"\n  [{num:>2}번]  {best_match}")
        print(f"         크기: {s_size:,.0f} KB  ↔  {data_files[best_match]['size']:,.0f} KB")
        print(f"         컬럼: {s_header[:5]}")
    else:
        print(f"\n  [{num:>2}번]  ❓ 매칭 실패  (크기: {s_size:,.0f} KB)")
        print(f"         컬럼: {s_header[:5]}")

print("\n" + "=" * 70)
print("data/ 파일 중 API 미제공 목록 (별도 업로드 필요)")
print("=" * 70)
for fname in sorted(data_files):
    if fname not in matched_data:
        print(f"  ❌  {fname}")

In [ ]:
# 정리: scan 폴더 삭제 여부 선택 (필요 시 주석 해제)
# shutil.rmtree(SCAN_DIR)
# print(f"{SCAN_DIR} 삭제 완료")